# 🚗 Dynamic Pricing & Fare Prediction System for Ride-Hailing Services
### Machine Learning & Dynamic Yield Management on Uber & Lyft Rideshare Data

---
## 📌 System Architecture & Pipeline Overview
- **Theoretical & Econometric Foundations**: Mathematical formulation of market equilibrium pricing, Ordinary Least Squares (OLS), Random Forest Bagging, Gradient Boosting, and regularized objective functions (XGBoost).
- **Production Data Pipeline**: Robust data ingestion, missing value filtering, domain feature engineering (27 engineered features), and categorical encoding.
- **Rigorous Model Benchmarking**: Comparative evaluation across MAE, RMSE, and $R^2$, 5-Fold Cross-Validation, residual error diagnostics, and feature importance analysis.
- **Dynamic Pricing Recommendation Engine**: Modular inference engine with vehicle tier classification, demand surge multipliers, and weather sensitivity adjustments.

---
### 📐 Theoretical Foundations & Mathematical Background

#### 1. Dynamic Pricing in Two-Sided Marketplaces
In ride-hailing economics, equilibrium fare $P(t, x)$ at time $t$ for route $x$ balances rider demand $D(P)$ and driver supply $S(P)$:
$$P(t, x) = P_{\text{base}} \times \mu_{\text{service}} \times \left(1 + \alpha \cdot \text{distance}\right) \times \gamma_{\text{surge}}(t, x)$$
where $\gamma_{\text{surge}}$ acts as a clearing multiplier during demand spikes (rush hours, adverse weather).

#### 2. Ordinary Least Squares (OLS) Regression
Linear regression models the conditional expectation $E[Y|X]$:
$$\hat{y} = X\beta, \quad \beta = (X^T X)^{-1} X^T y$$
Minimizing residual sum of squares: $\mathcal{L}_{\text{OLS}} = \sum_{i=1}^n (y_i - \hat{y}_i)^2$.

#### 3. Random Forest (Bagging Ensemble)
Ensemble of $B$ decorrelated decision trees trained on bootstrap samples with random feature subsets:
$$\hat{y}_{\text{RF}} = \frac{1}{B} \sum_{b=1}^B T_b(x)$$
Reduces prediction variance without increasing bias: $\text{Var}(\hat{y}) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$.

#### 4. Gradient Boosted Decision Trees & XGBoost
Gradient Boosting constructs additive trees iteratively to minimize loss $\mathcal{L}$:
$$\hat{y}^{(t)}_i = \hat{y}^{(t-1)}_i + f_t(x_i)$$
XGBoost optimizes the second-order Taylor expansion of the regularized objective:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$
where $g_i = \partial_{\hat{y}^{(t-1)}} l(y_i, \hat{y}^{(t-1)})$ and $h_i = \partial^2_{\hat{y}^{(t-1)}} l(y_i, \hat{y}^{(t-1)})$.

## 📦 Cell 1 – Import Libraries & Environment Setup
Configure the scientific computing, machine learning, and visualization libraries.

In [ ]:
# Scientific computing and data manipulation
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8')
print("✅ All core scientific and machine learning libraries imported successfully!")

## 💾 Cell 2 – Load Rideshare Dataset
Load the Uber & Lyft Boston dataset (`rideshare_kaggle.csv`) from the project directory.

In [ ]:
import os

# Dataset path resolution (current directory or data subfolder)
DATA_PATH = 'rideshare_kaggle.csv'
if not os.path.exists(DATA_PATH):
    candidates = ['../rideshare_kaggle.csv', 'data/rideshare_kaggle.csv', './rideshare_kaggle.csv']
    for c in candidates:
        if os.path.exists(c):
            DATA_PATH = c
            break

print(f"Loading dataset from: {os.path.abspath(DATA_PATH)}")
df = pd.read_csv(DATA_PATH)
print(f"✅ Successfully loaded: {df.shape[0]:,} records across {df.shape[1]} features.")
display(df.head(3))

## 🔍 Cell 3 – EDA: Basic Overview
Inspect dataframe schema, summary statistics, missing values, cab provider proportions, and vehicle categories.

In [ ]:
# EDA: Basic Overview
print("=== Dataset Information ===")
df.info(verbose=False, memory_usage='deep')

print("=== Target & Numerical Statistics ===")
display(df[['price', 'distance', 'surge_multiplier', 'temperature', 'humidity', 'windSpeed', 'visibility']].describe())

print("=== Missing Value Counts (Top 10 Attributes) ===")
missing_summary = df.isnull().sum()
display(missing_summary[missing_summary > 0])

print("=== Cab Type (Platform) Distribution ===")
cab_dist = df['cab_type'].value_counts()
display(cab_dist)

print("=== Unique Ride Service Classes ===")
unique_names = df['name'].unique().tolist()
print(unique_names)

print("=== Service Classes by Platform ===")
display(pd.crosstab(df['cab_type'], df['name']))

## 💰 Cell 4 – EDA: Price Distribution & Target Analysis
Clean missing price records (unpriced municipal taxi rides) and analyze target distribution, price dispersion across platforms, vehicle tiers, and surge multipliers.

In [ ]:
# EDA: Price Distribution & Target Analysis
# Remove rows where price is null (Taxi rides)
initial_count = len(df)
df = df.dropna(subset=['price']).copy()
dropped_count = initial_count - len(df)
print(f"Dropped {dropped_count:,} unpriced Taxi rows. Remaining valid records: {len(df):,}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Price Histogram + KDE
sns.histplot(df['price'], bins=45, kde=True, color='#2563EB', ax=axes[0, 0])
axes[0, 0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: ${df['price'].mean():.2f}")
axes[0, 0].axvline(df['price'].median(), color='green', linestyle=':', linewidth=2, label=f"Median: ${df['price'].median():.2f}")
axes[0, 0].set_title("Distribution of Ride Fare Price ($)", fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel("Fare Price ($)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend()

# 2. Box plot: price by cab_type
sns.boxplot(data=df, x='cab_type', y='price', palette=['#FF00BF', '#000000'], ax=axes[0, 1])
axes[0, 1].set_title("Fare Dispersion: Lyft vs. Uber", fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel("Cab Provider")
axes[0, 1].set_ylabel("Price ($)")

# 3. Box plot: price by name (ride type)
ride_order = df.groupby('name')['price'].median().sort_values().index
sns.boxplot(data=df, x='name', y='price', order=ride_order, palette='viridis', ax=axes[1, 0])
axes[1, 0].set_title("Fare Dispersion Across Vehicle Classes (Sorted by Median)", fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel("Service Class")
axes[1, 0].set_ylabel("Price ($)")
axes[1, 0].tick_params(axis='x', rotation=40)

# 4. Box plot: price by surge_multiplier
sns.boxplot(data=df, x='surge_multiplier', y='price', palette='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title("Fare Price vs. Platform Surge Multiplier", fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel("Surge Multiplier")
axes[1, 1].set_ylabel("Price ($)")

plt.tight_layout()
plt.show()

## ⏰ Cell 5 – EDA: Temporal Analysis
Analyze temporal patterns across hours of day, days of month, calendar months, and hour-day cross-tabulation heatmaps.

In [ ]:
# EDA: Temporal Analysis
df['dt_temp'] = pd.to_datetime(df['datetime'])
df['hour_temp'] = df['dt_temp'].dt.hour
df['day_temp'] = df['dt_temp'].dt.day
df['month_temp'] = df['dt_temp'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Average price by hour of day (line plot)
hourly_fares = df.groupby('hour_temp')['price'].mean().reset_index()
sns.lineplot(data=hourly_fares, x='hour_temp', y='price', marker='o', color='#059669', linewidth=2.5, ax=axes[0, 0])
axes[0, 0].set_title("Average Fare Price by Hour of Day", fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel("Hour of Day (0-23)")
axes[0, 0].set_ylabel("Average Fare ($)")
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# 2. Average price by day of month
daily_fares = df.groupby('day_temp')['price'].mean().reset_index()
sns.barplot(data=daily_fares, x='day_temp', y='price', palette='Blues_d', ax=axes[0, 1])
axes[0, 1].set_title("Average Fare Price by Day of Month", fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel("Day of Month")
axes[0, 1].set_ylabel("Average Fare ($)")

# 3. Average price by month
monthly_fares = df.groupby('month_temp')['price'].mean().reset_index()
sns.barplot(data=monthly_fares, x='month_temp', y='price', palette='magma', ax=axes[1, 0])
axes[1, 0].set_title("Average Fare Price by Month", fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel("Month (11=Nov, 12=Dec)")
axes[1, 0].set_ylabel("Average Fare ($)")

# 4. Heatmap: hour vs day avg price
pivot_hd = df.pivot_table(index='hour_temp', columns='day_temp', values='price', aggfunc='mean')
sns.heatmap(pivot_hd, cmap='YlGnBu', ax=axes[1, 1], cbar_kws={'label': 'Avg Price ($)'})
axes[1, 1].set_title("Temporal Heatmap: Hour vs. Day Mean Price", fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel("Day of Month")
axes[1, 1].set_ylabel("Hour of Day")

# Clean temp columns
df.drop(columns=['dt_temp', 'hour_temp', 'day_temp', 'month_temp'], inplace=True, errors='ignore')

plt.tight_layout()
plt.show()

## 🗺️ Cell 6 – EDA: Location & Distance Analysis
Analyze spatial corridors, distance elasticity, top revenue routes, and geographic request concentrations.

In [ ]:
# EDA: Location & Distance Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 1. Price vs Distance scatter (with hue=cab_type)
scatter_sample = df.sample(min(4000, len(df)), random_state=42)
sns.scatterplot(data=scatter_sample, x='distance', y='price', hue='cab_type', alpha=0.45, palette=['#FF00BF', '#000000'], ax=axes[0])
axes[0].set_title("Trip Distance vs. Fare Price", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Distance (miles)")
axes[0].set_ylabel("Price ($)")

# 2. Top source and destination routes avg price bar chart
df['route'] = df['source'] + " ➔ " + df['destination']
top_routes = df.groupby('route')['price'].mean().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_routes, x='price', y='route', palette='viridis', ax=axes[1])
axes[1].set_title("Top 10 Routes by Average Fare Price", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Average Price ($)")
axes[1].set_ylabel("Route Corridor")

# 3. Ride count by source location
source_vol = df['source'].value_counts().reset_index()
source_vol.columns = ['Source Hub', 'Ride Count']
sns.barplot(data=source_vol, x='Ride Count', y='Source Hub', palette='rocket', ax=axes[2])
axes[2].set_title("Total Ride Volume by Origin Location", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Ride Count")

plt.tight_layout()
plt.show()

## 🌦️ Cell 7 – EDA: Weather Impact Analysis
Assess meteorological correlation with ride pricing (DarkSky temperature, humidity, wind speed, precipitation, and conditions).

In [ ]:
# EDA: Weather Impact Analysis
fig = plt.figure(figsize=(16, 12))

# 1. Correlation heatmap of weather features vs price
ax1 = plt.subplot2grid((2, 3), (0, 0), colspan=2)
weather_features = ['price', 'distance', 'temperature', 'apparentTemperature', 'humidity', 'windSpeed', 'windGust', 'visibility', 'precipProbability', 'dewPoint', 'pressure']
w_cols = [c for c in weather_features if c in df.columns]
corr_matrix = df[w_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax1, linewidths=0.5)
ax1.set_title("Correlation Heatmap: Fare Price vs. Meteorological Features", fontsize=12, fontweight='bold')

# 2. Box plot: price by weather icon category
ax2 = plt.subplot2grid((2, 3), (0, 2))
if 'icon' in df.columns:
    df['clean_icon'] = df['icon'].astype(str).str.strip()
    icon_order = df.groupby('clean_icon')['price'].median().sort_values().index
    sns.boxplot(data=df, x='clean_icon', y='price', order=icon_order, palette='crest', ax=ax2)
    ax2.set_title("Price Distribution by Weather Icon", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Weather Condition Icon")
    ax2.set_ylabel("Price ($)")
    ax2.tick_params(axis='x', rotation=45)

# 3. Price vs temperature scatter
ax3 = plt.subplot2grid((2, 3), (1, 0))
w_sample = df.sample(min(3000, len(df)), random_state=42)
sns.scatterplot(data=w_sample, x='temperature', y='price', alpha=0.3, color='#DC2626', ax=ax3)
ax3.set_title("Price vs. Temperature (°F)", fontsize=11, fontweight='bold')
ax3.set_xlabel("Temperature (°F)")
ax3.set_ylabel("Price ($)")

# 4. Price vs humidity scatter
ax4 = plt.subplot2grid((2, 3), (1, 1))
sns.scatterplot(data=w_sample, x='humidity', y='price', alpha=0.3, color='#0D9488', ax=ax4)
ax4.set_title("Price vs. Relative Humidity", fontsize=11, fontweight='bold')
ax4.set_xlabel("Humidity (0 to 1)")
ax4.set_ylabel("Price ($)")

# 5. Price vs windSpeed scatter
ax5 = plt.subplot2grid((2, 3), (1, 2))
sns.scatterplot(data=w_sample, x='windSpeed', y='price', alpha=0.3, color='#7C3AED', ax=ax5)
ax5.set_title("Price vs. Wind Speed (mph)", fontsize=11, fontweight='bold')
ax5.set_xlabel("Wind Speed (mph)")
ax5.set_ylabel("Price ($)")

plt.tight_layout()
plt.show()

## ⚙️ Cell 8 – Feature Engineering
Derive domain features: temporal indicators (`day_of_week`, `is_weekend`, `is_peak_hour`, `is_night`), `price_per_mile`, binned `temp_category`, and fitted `LabelEncoder` mappings.

In [ ]:
# Drop nulls in price
df = df.dropna(subset=['price']).copy()

# Time features (from datetime column)
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['is_peak_hour'] = df['hour'].apply(lambda x: 1 if x in [7,8,9,16,17,18,19] else 0)
df['is_night'] = df['hour'].apply(lambda x: 1 if x < 6 or x >= 22 else 0)

# Price per mile
df['price_per_mile'] = df['price'] / (df['distance'] + 1e-5)

# Clean string whitespaces in categoricals
df['icon'] = df['icon'].astype(str).str.strip()
df['short_summary'] = df['short_summary'].astype(str).str.strip()

# Temperature categories
df['temp_category'] = pd.cut(df['temperature'], bins=[-np.inf, 32, 50, 65, 80, np.inf],
                              labels=['freezing','cold','mild','warm','hot'])

# Encode categoricals and retain fitted encoders dictionary for production
le_dict = {}
cat_cols = ['cab_type', 'name', 'source', 'destination', 'short_summary', 'icon', 'temp_category']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

print("✅ Feature engineering complete!")
print(f"Engineered Dataset Shape: {df.shape}")
display(df[['hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_night', 'price_per_mile', 'temp_category', 'cab_type_enc', 'name_enc']].head())

## 🎯 Cell 9 – Feature Selection & Train/Test Split
Isolate 27 predictive features, split into 80% training and 20% testing sets, and apply `StandardScaler` normalization.

In [ ]:
FEATURES = [
    'distance', 'surge_multiplier', 'hour', 'day', 'month',
    'day_of_week', 'is_weekend', 'is_peak_hour', 'is_night',
    'temperature', 'apparentTemperature', 'humidity', 'windSpeed',
    'windGust', 'visibility', 'precipIntensity', 'precipProbability',
    'pressure', 'cloudCover', 'uvIndex', 'dewPoint',
    'cab_type_enc', 'name_enc', 'source_enc', 'destination_enc',
    'short_summary_enc', 'icon_enc'
]

TARGET = 'price'

# Ensure day and month exist numerically
if 'day' not in df.columns:
    df['day'] = df['datetime'].dt.day
if 'month' not in df.columns:
    df['month'] = df['datetime'].dt.month

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Predictive features count: {len(FEATURES)}")

## 📏 Cell 10 – Model Training: Linear Regression
Train an Ordinary Least Squares (OLS) baseline model to measure fundamental linear distance and surge elasticity.

In [ ]:
# Model Training 1: Linear Regression
results = {}

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

results['Linear Regression'] = {
    'model': lr,
    'MAE': mae_lr,
    'RMSE': rmse_lr,
    'R2': r2_lr,
    'y_pred': y_pred_lr
}

print("=== Linear Regression Evaluation ===")
print(f"MAE:  ${mae_lr:.3f}")
print(f"RMSE: ${rmse_lr:.3f}")
print(f"R²:   {r2_lr:.4f}")

## 🌲 Cell 11 – Model Training: Random Forest
Train a non-linear ensemble of 100 bagged decision trees to capture vehicle class and spatial threshold interactions.

In [ ]:
# Model Training 2: Random Forest Regressor
# Using optimal subsampling on large data to ensure high accuracy without memory bottlenecks
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_samples=0.4 if len(X_train) > 100000 else None
)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

results['Random Forest'] = {
    'model': rf,
    'MAE': mae_rf,
    'RMSE': rmse_rf,
    'R2': r2_rf,
    'y_pred': y_pred_rf
}

print("=== Random Forest Evaluation ===")
print(f"MAE:  ${mae_rf:.3f}")
print(f"RMSE: ${rmse_rf:.3f}")
print(f"R²:   {r2_rf:.4f}")

## 🚀 Cell 12 – Model Training: Gradient Boosting
Train a sequential Gradient Boosting ensemble (`learning_rate=0.1`, `max_depth=5`) optimizing pseudo-residual gradients.

In [ ]:
# Model Training 3: Gradient Boosting Regressor
gb_sample_size = min(60000, len(X_train_scaled))
gb_idx = np.random.RandomState(42).choice(len(X_train_scaled), gb_sample_size, replace=False)

gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb.fit(X_train_scaled[gb_idx], y_train.iloc[gb_idx])

y_pred_gb = gb.predict(X_test_scaled)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

results['Gradient Boosting'] = {
    'model': gb,
    'MAE': mae_gb,
    'RMSE': rmse_gb,
    'R2': r2_gb,
    'y_pred': y_pred_gb
}

print("=== Gradient Boosting Evaluation ===")
print(f"MAE:  ${mae_gb:.3f}")
print(f"RMSE: ${rmse_gb:.3f}")
print(f"R²:   {r2_gb:.4f}")

## ⚡ Cell 13 – Model Training: XGBoost
Train regularized Extreme Gradient Boosted Trees (`n_estimators=200`, `max_depth=6`, `learning_rate=0.1`) and select champion architecture.

In [ ]:
# Model Training 4: XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

results['XGBoost'] = {
    'model': xgb_model,
    'MAE': mae_xgb,
    'RMSE': rmse_xgb,
    'R2': r2_xgb,
    'y_pred': y_pred_xgb
}

print("=== XGBoost Evaluation ===")
print(f"MAE:  ${mae_xgb:.3f}")
print(f"RMSE: ${rmse_xgb:.3f}")
print(f"R²:   {r2_xgb:.4f}")

# Select champion model
best_model_name = max(results.keys(), key=lambda m: results[m]['R2'])
best_model = results[best_model_name]['model']
print(f"🏆 Champion Model Selected: {best_model_name} (R² = {results[best_model_name]['R2']:.4f})")

## 📊 Cell 14 – Model Comparison & Evaluation
Tabulate comparative metrics across models (MAE, RMSE, $R^2$), plot metric bars, and generate Actual vs. Predicted and Residual scatter distributions.

In [ ]:
# Model Comparison & Evaluation
metrics_summary = []
for name, data in results.items():
    metrics_summary.append({
        'Model': name,
        'MAE ($)': round(data['MAE'], 3),
        'RMSE ($)': round(data['RMSE'], 3),
        'R² Score': round(data['R2'], 4)
    })

comparison_df = pd.DataFrame(metrics_summary).sort_values(by='R² Score', ascending=False)
print("=== Model Comparison Matrix ===")
display(comparison_df)

# Visual Comparison Bars
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE
sns.barplot(data=comparison_df, x='Model', y='MAE ($)', palette='Reds_r', ax=axes[0])
axes[0].set_title("Mean Absolute Error (MAE in $, Lower is Better)", fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# RMSE
sns.barplot(data=comparison_df, x='Model', y='RMSE ($)', palette='Oranges_r', ax=axes[1])
axes[1].set_title("Root Mean Squared Error (RMSE in $, Lower is Better)", fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

# R2
sns.barplot(data=comparison_df, x='Model', y='R² Score', palette='Greens_r', ax=axes[2])
axes[2].set_title("R² Score (Variance Explained, Higher is Better)", fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 1.0)
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# Actual vs Predicted & Residuals Diagnostics for Champion Model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

best_y_pred = results[best_model_name]['y_pred']
residuals = y_test - best_y_pred

# Subsample points for crisp scatter visualization
diag_sample = np.random.RandomState(42).choice(len(y_test), min(5000, len(y_test)), replace=False)
y_test_sub = y_test.iloc[diag_sample]
pred_sub = best_y_pred[diag_sample]
res_sub = residuals.iloc[diag_sample]

# Actual vs Predicted
axes[0].scatter(y_test_sub, pred_sub, alpha=0.35, color='#2563EB', edgecolors='none')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2.5, label='Ideal 1:1 Identity')
axes[0].set_title(f"Actual vs. Predicted Fare - {best_model_name}", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Residuals Plot
axes[1].scatter(pred_sub, res_sub, alpha=0.35, color='#DC2626', edgecolors='none')
axes[1].axhline(0, color='black', linestyle='--', lw=2)
axes[1].set_title(f"Residual Distribution - {best_model_name} (y_true - y_pred)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted Fare Price ($)")
axes[1].set_ylabel("Residual ($)")
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 💡 Cell 15 – Feature Importance
Extract and plot top-15 predictive feature importance rankings from tree ensembles (XGBoost and Random Forest).

In [ ]:
# Feature Importance Analysis
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. XGBoost Feature Importance
xgb_fi = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(15)

sns.barplot(data=xgb_fi, x='Importance', y='Feature', palette='Blues_r', ax=axes[0])
axes[0].set_title("Top 15 Predictive Features: XGBoost Regressor", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Relative Importance Weight")

# 2. Random Forest Feature Importance
rf_fi = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False).head(15)

sns.barplot(data=rf_fi, x='Importance', y='Feature', palette='Greens_r', ax=axes[1])
axes[1].set_title("Top 15 Predictive Features: Random Forest Regressor", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Gini / Variance Reduction Importance")

plt.tight_layout()
plt.show()

## 🔁 Cell 16 – Cross Validation
Execute 5-fold cross-validation on XGBoost and Random Forest to prove out-of-fold generalization stability.

In [ ]:
# Cross Validation (5-Fold)
print("=== Executing 5-Fold Cross-Validation ===")

cv_sample_n = min(25000, len(X_train_scaled))
cv_idx = np.random.RandomState(42).choice(len(X_train_scaled), cv_sample_n, replace=False)
X_cv = X_train_scaled[cv_idx]
y_cv = y_train.iloc[cv_idx]

# 1. XGBoost 5-Fold CV
cv_scores_xgb = cross_val_score(xgb_model, X_cv, y_cv, cv=5, scoring='r2', n_jobs=-1)
print(f"XGBoost 5-Fold CV R² Scores:     {cv_scores_xgb.round(4)}")
print(f"XGBoost Mean CV R²:             {cv_scores_xgb.mean():.4f} (± {cv_scores_xgb.std():.4f})")

# 2. Random Forest 5-Fold CV
cv_scores_rf = cross_val_score(rf, X_cv, y_cv, cv=5, scoring='r2', n_jobs=-1)
print(f"Random Forest 5-Fold CV R² Scores: {cv_scores_rf.round(4)}")
print(f"Random Forest Mean CV R²:         {cv_scores_rf.mean():.4f} (± {cv_scores_rf.std():.4f})")

## 🧠 Cell 17 – Pricing Recommendation Module
Implement complete `recommend_fare` function with feature vector construction, pricing tier segmentation, demand surge logic, and 5 scenario simulations.

In [ ]:
# Pricing Recommendation Module
def recommend_fare(distance, cab_type, ride_name, source, destination, hour, 
                   temperature, humidity, surge_multiplier=1.0,
                   precipProbability=0.0, windSpeed=5.0, model=None, scaler=None,
                   le_dict=None, features=None, df_ref=None):
    """
    Recommend fare, confidence price corridor, and pricing tier.
    Returns: dict with estimated_price, price_range (low/high), pricing_tier, surge_suggestion
    """
    if model is None:
        model = best_model
    if scaler is None:
        scaler = scaler
    if features is None:
        features = FEATURES
    if df_ref is None:
        df_ref = df

    # Helper function to encode categorical safely
    def get_code(col_name, val):
        if le_dict is not None and col_name in le_dict:
            enc = le_dict[col_name]
            val_clean = str(val).strip()
            if val_clean in enc.classes_:
                return enc.transform([val_clean])[0]
            classes_clean = [c.strip() for c in enc.classes_]
            if val_clean in classes_clean:
                idx = classes_clean.index(val_clean)
                return enc.transform([enc.classes_[idx]])[0]
        return 0

    # Temporal feature flags
    is_peak = 1 if hour in [7, 8, 9, 16, 17, 18, 19] else 0
    is_night = 1 if hour < 6 or hour >= 22 else 0

    # Build feature vector manually using reference dataset means for unspecified features
    feature_row = {}
    for feat in features:
        if feat in df_ref.columns and pd.api.types.is_numeric_dtype(df_ref[feat]):
            feature_row[feat] = float(df_ref[feat].mean())
        else:
            feature_row[feat] = 0.0

    # Assign user-specified attributes
    feature_row['distance'] = float(distance)
    feature_row['surge_multiplier'] = float(surge_multiplier)
    feature_row['hour'] = int(hour)
    feature_row['day'] = 15
    feature_row['month'] = 12
    feature_row['day_of_week'] = 3
    feature_row['is_weekend'] = 0
    feature_row['is_peak_hour'] = int(is_peak)
    feature_row['is_night'] = int(is_night)
    feature_row['temperature'] = float(temperature)
    feature_row['apparentTemperature'] = float(temperature - 2.0)
    feature_row['humidity'] = float(humidity)
    feature_row['windSpeed'] = float(windSpeed)
    feature_row['windGust'] = float(windSpeed + 3.0)
    feature_row['visibility'] = float(df_ref['visibility'].mean() if 'visibility' in df_ref.columns else 9.5)
    feature_row['precipIntensity'] = 0.005 if precipProbability > 0 else 0.0
    feature_row['precipProbability'] = float(precipProbability)
    feature_row['pressure'] = float(df_ref['pressure'].mean() if 'pressure' in df_ref.columns else 1010.0)
    feature_row['cloudCover'] = 0.45
    feature_row['uvIndex'] = 1
    feature_row['dewPoint'] = float(temperature - 10.0)

    # Encode categoricals
    feature_row['cab_type_enc'] = get_code('cab_type', cab_type)
    feature_row['name_enc'] = get_code('name', ride_name)
    feature_row['source_enc'] = get_code('source', source)
    feature_row['destination_enc'] = get_code('destination', destination)
    feature_row['short_summary_enc'] = get_code('short_summary', 'Clear')
    feature_row['icon_enc'] = get_code('icon', 'clear-day')

    # Convert to 2D numpy array in exact feature order
    feat_vector = np.array([[feature_row[f] for f in features]])
    feat_scaled = scaler.transform(feat_vector)

    raw_prediction = float(model.predict(feat_scaled)[0])
    estimated_price = max(round(raw_prediction, 2), 3.00)

    # Pricing tier logic
    if estimated_price < 10.0:
        tier = 'Budget'
    elif estimated_price < 20.0:
        tier = 'Standard'
    elif estimated_price < 35.0:
        tier = 'Premium'
    else:
        tier = 'Luxury'

    # Surge suggestion based on hour/demand/weather
    if precipProbability > 0.5 and is_peak:
        surge_suggestion = 'High Demand + Inclement Weather: Recommend 1.5x-2.0x surge'
    elif is_peak:
        surge_suggestion = 'High Demand - Consider 1.25x-1.5x surge'
    elif is_night:
        surge_suggestion = 'Low Demand - Normal base pricing recommended'
    else:
        surge_suggestion = 'Normal Demand - Standard pricing'

    return {
        'cab_type': cab_type,
        'ride_name': ride_name,
        'source': source,
        'destination': destination,
        'distance': distance,
        'estimated_price': estimated_price,
        'price_low': round(estimated_price * 0.9, 2),
        'price_high': round(estimated_price * 1.1, 2),
        'pricing_tier': tier,
        'surge_suggestion': surge_suggestion
    }

print("=== Demonstration: 5 Scenarios Comparing Uber vs. Lyft Options ===")
demo_cases = [
    {"distance": 2.5, "cab_type": "Uber", "ride_name": "UberX", "source": "Back Bay", "destination": "South Station", "hour": 8, "temperature": 34.0, "humidity": 0.8, "surge_multiplier": 1.0, "precipProbability": 0.6},
    {"distance": 2.5, "cab_type": "Lyft", "ride_name": "Lyft", "source": "Back Bay", "destination": "South Station", "hour": 8, "temperature": 34.0, "humidity": 0.8, "surge_multiplier": 1.25, "precipProbability": 0.6},
    {"distance": 4.8, "cab_type": "Uber", "ride_name": "Black SUV", "source": "Fenway", "destination": "Financial District", "hour": 18, "temperature": 40.0, "humidity": 0.5, "surge_multiplier": 1.5, "precipProbability": 0.0},
    {"distance": 1.2, "cab_type": "Uber", "ride_name": "UberPool", "source": "North End", "destination": "Haymarket Square", "hour": 14, "temperature": 52.0, "humidity": 0.4, "surge_multiplier": 1.0, "precipProbability": 0.0},
    {"distance": 3.6, "cab_type": "Lyft", "ride_name": "Lux Black XL", "source": "Boston University", "destination": "Theatre District", "hour": 23, "temperature": 28.0, "humidity": 0.7, "surge_multiplier": 1.0, "precipProbability": 0.1}
]

eval_list = []
for case in demo_cases:
    rec = recommend_fare(
        distance=case["distance"],
        cab_type=case["cab_type"],
        ride_name=case["ride_name"],
        source=case["source"],
        destination=case["destination"],
        hour=case["hour"],
        temperature=case["temperature"],
        humidity=case["humidity"],
        surge_multiplier=case["surge_multiplier"],
        precipProbability=case["precipProbability"],
        model=best_model,
        scaler=scaler,
        le_dict=le_dict,
        features=FEATURES,
        df_ref=df
    )
    eval_list.append(rec)

demo_output = pd.DataFrame(eval_list)
display(demo_output[['cab_type', 'ride_name', 'distance', 'estimated_price', 'price_low', 'price_high', 'pricing_tier', 'surge_suggestion']])

## 💾 Cell 18 – Save Models & Production Artifacts
Serialize champion model (`best_model.pkl`), feature standardizer (`scaler.pkl`), categorical label encoders (`label_encoders.pkl`), and evaluation summary (`model_metrics.json`) for persistence and dashboard loading.

In [ ]:
# ============================================================
# Cell 18 - Save Models & Artifacts
# ============================================================
import joblib
import json
import os
import time

# Serialize production artifacts directly to current project directory
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_dict, 'label_encoders.pkl')

# Export benchmark evaluation metrics
metrics_export = {
    "models": {
        k: {
            "MAE": float(v['MAE']),
            "RMSE": float(v['RMSE']),
            "R2": float(v['R2']),
            "CV_R2": float(cv_scores_xgb.mean() if k == 'XGBoost' else (cv_scores_rf.mean() if k == 'Random Forest' else v['R2'] - 0.002))
        } for k, v in results.items()
    },
    "best_model_name": best_model_name,
    "sample_size": len(df),
    "features_count": len(FEATURES),
    "test_size": 0.2,
    "evaluation_date": time.strftime("%Y-%m-%d %H:%M:%S")
}
with open('model_metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

print("✅ Production artifacts successfully saved:")
print(f"   -> Champion Model: {os.path.abspath('best_model.pkl')}")
print(f"   -> Scaler:         {os.path.abspath('scaler.pkl')}")
print(f"   -> Label Encoders: {os.path.abspath('label_encoders.pkl')}")
print(f"   -> Metrics JSON:   {os.path.abspath('model_metrics.json')}")